In [1]:
import pyspark
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/23 14:49:29 WARN Utils: Your hostname, DESKTOP-JCM8NP2, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/08/23 14:49:29 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/23 14:49:31 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
df_green = spark.read \
    .option("header", "true") \
    .csv('data/raw/green/2021/01/')

In [4]:
import pandas as pd

In [5]:
df_green_pd = pd.read_csv('data/raw/green/2021/01/green_tripdata_2021-01.csv.gz', nrows=1000)

In [8]:
from pyspark.sql import types

In [12]:
green_schema = types.StructType([
    types.StructField("VendorID", types.IntegerType(), True), 
    types.StructField("lpep_pickup_datetime", types.TimestampType(), True), 
    types.StructField("lpep_dropoff_datetime", types.TimestampType(), True), 
    types.StructField("store_and_fwd_flag", types.StringType(), True), 
    types.StructField("RatecodeID", types.IntegerType(), True), 
    types.StructField("PULocationID", types.IntegerType(), True), 
    types.StructField("DOLocationID", types.IntegerType(), True), 
    types.StructField("passenger_count", types.IntegerType(), True), 
    types.StructField("trip_distance", types.DoubleType(), True), 
    types.StructField("fare_amount", types.DoubleType(), True), 
    types.StructField("extra", types.DoubleType(), True), 
    types.StructField("mta_tax", types.DoubleType(), True), 
    types.StructField("tip_amount", types.DoubleType(), True), 
    types.StructField("tolls_amount", types.DoubleType(), True), 
    types.StructField("ehail_fee", types.DoubleType(), True), 
    types.StructField("improvement_surcharge", types.DoubleType(), True), 
    types.StructField("total_amount", types.DoubleType(), True), 
    types.StructField("payment_type", types.IntegerType(), True), 
    types.StructField("trip_type", types.IntegerType(), True), 
    types.StructField("congestion_surcharge", types.DoubleType(), True)
])

In [14]:
df_green.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- lpep_pickup_datetime: timestamp (nullable = true)
 |-- lpep_dropoff_datetime: timestamp (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- RatecodeID: integer (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- ehail_fee: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- trip_type: integer (nullable = true)
 |-- congestion_surcharge: double (nullable = true)



In [19]:
year = 2021

for month in range(1, 13):
    print(f'processing data for {year}/{month}')
    input_path = f'data/raw/green/{year}/{month:02d}/'
    output_path = f'data/pq/green/{year}/{month:02d}/'

    df_green = spark.read \
        .option("header", "true") \
        .schema(green_schema) \
        .csv(input_path)
    df_green \
        .repartition(4) \
        .write.parquet(output_path)

processing data for 2021/1


processing data for 2021/2
processing data for 2021/3
processing data for 2021/4
processing data for 2021/5
processing data for 2021/6
processing data for 2021/7
processing data for 2021/8
processing data for 2021/9


26/08/23 16:03:29 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: data/raw/green/2021/09/.
java.io.FileNotFoundException: File data/raw/green/2021/09 does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:394)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:210)
	at org.apache.spark.sql.catalyst.analysis.

AnalysisException: [PATH_NOT_FOUND] Path does not exist: file:/home/eclavel/projects/learning/DE_ZC_HW/week_5_spark/notebooks/data/raw/green/2021/09. SQLSTATE: 42K03

In [15]:
df_yellow_pd = pd.read_csv('data/raw/yellow/2021/01/yellow_tripdata_2021-01.csv.gz', nrows=1000)

In [16]:
spark.createDataFrame(df_yellow_pd).schema

StructType([StructField('VendorID', LongType(), True), StructField('tpep_pickup_datetime', StringType(), True), StructField('tpep_dropoff_datetime', StringType(), True), StructField('passenger_count', LongType(), True), StructField('trip_distance', DoubleType(), True), StructField('RatecodeID', LongType(), True), StructField('store_and_fwd_flag', StringType(), True), StructField('PULocationID', LongType(), True), StructField('DOLocationID', LongType(), True), StructField('payment_type', LongType(), True), StructField('fare_amount', DoubleType(), True), StructField('extra', DoubleType(), True), StructField('mta_tax', DoubleType(), True), StructField('tip_amount', DoubleType(), True), StructField('tolls_amount', DoubleType(), True), StructField('improvement_surcharge', DoubleType(), True), StructField('total_amount', DoubleType(), True), StructField('congestion_surcharge', DoubleType(), True)])

In [17]:
yellow_schema = types.StructType([
    types.StructField("VendorID", types.IntegerType(), True), 
    types.StructField("tpep_pickup_datetime", types.TimestampType(), True), 
    types.StructField("tpep_dropoff_datetime", types.TimestampType(), True), 
    types.StructField("passenger_count", types.IntegerType(), True), 
    types.StructField("trip_distance", types.DoubleType(), True), 
    types.StructField("RatecodeID", types.IntegerType(), True), 
    types.StructField("store_and_fwd_flag", types.StringType(), True), 
    types.StructField("PULocationID", types.IntegerType(), True), 
    types.StructField("DOLocationID", types.IntegerType(), True), 
    types.StructField("payment_type", types.IntegerType(), True), 
    types.StructField("fare_amount", types.DoubleType(), True), 
    types.StructField("extra", types.DoubleType(), True), 
    types.StructField("mta_tax", types.DoubleType(), True), 
    types.StructField("tip_amount", types.DoubleType(), True), 
    types.StructField("tolls_amount", types.DoubleType(), True), 
    types.StructField("improvement_surcharge", types.DoubleType(), True), 
    types.StructField("total_amount", types.DoubleType(), True), 
    types.StructField("congestion_surcharge", types.DoubleType(), True)
])

In [18]:
df_yellow = spark.read \
    .option("header", "true") \
    .schema(yellow_schema) \
    .csv('data/raw/yellow/2021/01/')

In [20]:
year = 2021

for month in range(1, 13):
    print(f'processing data for {year}/{month}')
    input_path = f'data/raw/yellow/{year}/{month:02d}/'
    output_path = f'data/pq/yellow/{year}/{month:02d}/'

    df_yellow = spark.read \
        .option("header", "true") \
        .schema(yellow_schema) \
        .csv(input_path)
    df_yellow \
        .repartition(4) \
        .write.parquet(output_path)

processing data for 2021/1


processing data for 2021/2


processing data for 2021/3


processing data for 2021/4


processing data for 2021/5


processing data for 2021/6


processing data for 2021/7


processing data for 2021/8
processing data for 2021/9


26/08/23 16:08:20 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: data/raw/yellow/2021/09/.
java.io.FileNotFoundException: File data/raw/yellow/2021/09 does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:394)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:210)
	at org.apache.spark.sql.catalyst.analysi

AnalysisException: [PATH_NOT_FOUND] Path does not exist: file:/home/eclavel/projects/learning/DE_ZC_HW/week_5_spark/notebooks/data/raw/yellow/2021/09. SQLSTATE: 42K03

In [6]:
spark.createDataFrame(df_green_pd).schema

StructType([StructField('VendorID', LongType(), True), StructField('lpep_pickup_datetime', StringType(), True), StructField('lpep_dropoff_datetime', StringType(), True), StructField('store_and_fwd_flag', StringType(), True), StructField('RatecodeID', LongType(), True), StructField('PULocationID', LongType(), True), StructField('DOLocationID', LongType(), True), StructField('passenger_count', LongType(), True), StructField('trip_distance', DoubleType(), True), StructField('fare_amount', DoubleType(), True), StructField('extra', DoubleType(), True), StructField('mta_tax', DoubleType(), True), StructField('tip_amount', DoubleType(), True), StructField('tolls_amount', DoubleType(), True), StructField('ehail_fee', DoubleType(), True), StructField('improvement_surcharge', DoubleType(), True), StructField('total_amount', DoubleType(), True), StructField('payment_type', LongType(), True), StructField('trip_type', LongType(), True), StructField('congestion_surcharge', DoubleType(), True)])

In [7]:
df_green.printSchema()

root
 |-- VendorID: string (nullable = true)
 |-- lpep_pickup_datetime: string (nullable = true)
 |-- lpep_dropoff_datetime: string (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- RatecodeID: string (nullable = true)
 |-- PULocationID: string (nullable = true)
 |-- DOLocationID: string (nullable = true)
 |-- passenger_count: string (nullable = true)
 |-- trip_distance: string (nullable = true)
 |-- fare_amount: string (nullable = true)
 |-- extra: string (nullable = true)
 |-- mta_tax: string (nullable = true)
 |-- tip_amount: string (nullable = true)
 |-- tolls_amount: string (nullable = true)
 |-- ehail_fee: string (nullable = true)
 |-- improvement_surcharge: string (nullable = true)
 |-- total_amount: string (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- trip_type: string (nullable = true)
 |-- congestion_surcharge: string (nullable = true)

